<a href="https://colab.research.google.com/github/Dulandi-Dinethya/SE4050-Deep_Learning_Assignment/blob/IT23185302/results/models/02_VGG16_IT23185302.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VGG16-Based Palmprint Recognition

**Module:** SE4050 – Deep Learning  
**Student ID:** IT23185302  
**Model:** VGG16  

## Project
**Deep Learning-Based Palmprint Recognition System for Contactless Biometric Identification and Authentication of Registered Users**

## Objective
This notebook implements and evaluates a VGG16-based deep learning model for palmprint recognition. Transfer learning with a VGG16 model pretrained on ImageNet is used to extract visual features from palm images and classify them according to registered user identities.

## 1. Environment Setup and Library Imports

The required Python libraries for data handling, visualization, deep learning, and model evaluation are imported in this section. TensorFlow/Keras is used to implement the VGG16 model, while scikit-learn is used to calculate classification performance metrics.

In [1]:
# Basic libraries
import os
import random
import time

# Data handling
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# VGG16 model
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

# Evaluation metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


### 1.1 Reproducibility

A fixed random seed is used to improve the reproducibility of the experiment.

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Random seed:", SEED)

Random seed: 42


## 2. Dataset and Prepared Data Loading

The palm images were preprocessed using the common preprocessing
pipeline developed for the project. The successfully processed images
from Session 1 and Session 2 are loaded for the VGG16 experiment.
Samples that failed ROI extraction during preprocessing are excluded
from model training.

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
DATA_DRIVE = "/content/drive/MyDrive/SE4050_Palm_Dataset"

SESSION1_ZIP = os.path.join(DATA_DRIVE, "session1.zip")
SESSION2_ZIP = os.path.join(DATA_DRIVE, "session2.zip")

print("Session 1 ZIP exists:", os.path.exists(SESSION1_ZIP))
print("Session 2 ZIP exists:", os.path.exists(SESSION2_ZIP))

Session 1 ZIP exists: True
Session 2 ZIP exists: True


In [5]:
import zipfile

EXTRACT_DIR = "/content/palm_preprocessed"

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(SESSION1_ZIP, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

with zipfile.ZipFile(SESSION2_ZIP, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Dataset extraction completed.")

Dataset extraction completed.


In [6]:
for root, dirs, files in os.walk(EXTRACT_DIR):
    level = root.replace(EXTRACT_DIR, "").count(os.sep)
    indent = "    " * level

    print(
        f"{indent}{os.path.basename(root)}/ "
        f"({len(files)} files, {len(dirs)} folders)"
    )

    if level >= 2:
        dirs[:] = []

palm_preprocessed/ (0 files, 2 folders)
    session1/ (5994 files, 0 folders)
    session2/ (5994 files, 0 folders)


In [7]:
image_extensions = (".png", ".jpg", ".jpeg", ".tiff", ".tif")

all_images = []

for root, dirs, files in os.walk(EXTRACT_DIR):
    for file in files:
        if file.lower().endswith(image_extensions):
            all_images.append(os.path.join(root, file))

print("Total usable image files found:", len(all_images))
print("\nFirst 10 files:")

for path in all_images[:10]:
    print(path)

Total usable image files found: 11988

First 10 files:
/content/palm_preprocessed/session1/05157.png
/content/palm_preprocessed/session1/00085.png
/content/palm_preprocessed/session1/04931.png
/content/palm_preprocessed/session1/03848.png
/content/palm_preprocessed/session1/05429.png
/content/palm_preprocessed/session1/04035.png
/content/palm_preprocessed/session1/01482.png
/content/palm_preprocessed/session1/02441.png
/content/palm_preprocessed/session1/04835.png
/content/palm_preprocessed/session1/04520.png
